# Smoke test: MedViTV2-tiny on DermaMNIST (ELEC-444)
Colab now uses Python 3.13, but this model needs Python 3.11 + PyTorch 2.5 + NATTEN 0.17.
So this notebook builds a separate Python 3.11 setup inside Colab and runs the test with it.

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save.
Run the cells **one at a time, top to bottom**. No restart needed.

## Cell 1: Check the GPU

In [ ]:
# ===== CELL 1 =====
!nvidia-smi
print('\n>>> CELL 1 DONE. Now run Cell 2.')

## Cell 2: Connect Google Drive (results are saved there)
A popup asks for permission: pick your account and click Continue / Allow.

In [ ]:
# ===== CELL 2 =====
from google.colab import drive
drive.mount('/content/drive')
print('\n>>> CELL 2 DONE. Now run Cell 3.')

## Cell 3: Build the Python 3.11 setup (5–10 minutes)
Lots of text is normal. Wait for the DONE line.

In [ ]:
# ===== CELL 3 =====
!pip install -q uv
!uv venv --python 3.11 /content/py311 -q
!uv pip install -q --python /content/py311/bin/python torch==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu124
!uv pip install -q --python /content/py311/bin/python "natten==0.17.5+torch250cu124" -f https://whl.natten.org
!uv pip install -q --python /content/py311/bin/python timm==1.0.15 einops medmnist==3.0.2 numpy pandas scikit-learn scikit-image tqdm pillow gdown
![ -d /content/MedViTV2 ] || git clone -q https://github.com/Omid-Nejati/MedViTV2.git /content/MedViTV2
# Use the model file version that matches NATTEN 0.17 and the released checkpoints (before the Oct 23, 2026 update)
!cd /content/MedViTV2 && git checkout -q 4a9e517 -- MedViT.py
print('\nChecking versions with the new Python:')
!/content/py311/bin/python -c "import sys, torch, natten; print('python', sys.version.split()[0], '| torch', torch.__version__, '| natten', natten.__version__, '| GPU ok:', torch.cuda.is_available())"
print('\n>>> CELL 3 DONE if the line above shows python 3.11, torch 2.5.0, natten 0.17.5 and GPU ok: True. Now run Cell 4.')

## Cell 4: Save the test script
Just prints `Writing /content/smoke_test.py`. Takes a second.

In [ ]:
%%writefile /content/smoke_test.py
"""Smoke test for MedViTV2-tiny on DermaMNIST (ELEC-444). Run with the Python 3.11 env."""
import os, sys, time
sys.path.insert(0, '/content/MedViTV2')
os.chdir('/content/MedViTV2')
os.makedirs('./data', exist_ok=True)  # medmnist needs this folder to exist

import numpy as np, torch, torch.nn as nn, natten, medmnist, gdown
import torchvision.transforms as T
from medmnist import INFO
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             roc_auc_score, confusion_matrix)
from MedViT import MedViT_tiny

OUT = '/content/drive/MyDrive/elec444_project/smoke_test'
os.makedirs(OUT, exist_ok=True)
device = 'cuda'
print('python', sys.version.split()[0], '| torch', torch.__version__, '| natten', natten.__version__,
      '| GPU:', torch.cuda.get_device_name(0))

# ---------- PART 1: data ----------
print('\n========== PART 1: DATASET ==========')
info = INFO['dermamnist']
CLASSES = [info['label'][str(i)] for i in range(len(info['label']))]
DataClass = getattr(medmnist, info['python_class'])
test_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(mean=[.5], std=[.5])])
train_tf = T.Compose([T.RandomResizedCrop(224), T.AugMix(alpha=0.4), T.RandomHorizontalFlip(p=0.4),
                      T.ToTensor(), T.Normalize(mean=[.5], std=[.5])])  # same as repo
ds = {s: DataClass(split=s, transform=(train_tf if s == 'train' else test_tf), download=True,
                   as_rgb=True, root='./data', size=224, mmap_mode='r') for s in ['train', 'val', 'test']}
for s, d in ds.items():
    print(f'{s:5s} n={len(d):5d}  per class: {np.bincount(d.labels.flatten(), minlength=len(CLASSES)).tolist()}')
for i, c in enumerate(CLASSES):
    print(i, c)

# ---------- PART 2: released checkpoint ----------
print('\n========== PART 2: RELEASED CHECKPOINT (README reports 78.1%) ==========')
CKPT = 'dermamnist_MedViT_tiny.pth'
if not os.path.exists(CKPT):
    gdown.download(id='1kFsxcB0L6S_WiEayDdyFoviXw8QInT71', output=CKPT, quiet=True)
net = MedViT_tiny(num_classes=len(CLASSES)).to(device)
ck = torch.load(CKPT, map_location=device)
sd = ck['model'] if isinstance(ck, dict) and 'model' in ck else ck
missing, unexpected = net.load_state_dict(sd, strict=False)
print('missing keys:', len(missing), '| unexpected keys:', len(unexpected))
if isinstance(ck, dict) and 'acc' in ck:
    print('accuracy stored in checkpoint:', ck['acc'], '| epoch:', ck.get('epoch'))

@torch.no_grad()
def predict(model, dataset, bs=128):
    model.eval()
    probs, ys = [], []
    for x, y in DataLoader(dataset, batch_size=bs, shuffle=False, num_workers=2):
        probs.append(model(x.to(device)).softmax(-1).cpu())
        ys.append(y.view(-1))
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

probs, y = predict(net, ds['test'])
pred = probs.argmax(1)
acc = accuracy_score(y, pred)
report = classification_report(y, pred, target_names=CLASSES, digits=3, zero_division=0)
cm = confusion_matrix(y, pred)
print(f'Test accuracy:      {acc:.4f}   (README reports 0.781)')
print(f'Balanced accuracy:  {balanced_accuracy_score(y, pred):.4f}')
print(f'Macro AUC (OvR):    {roc_auc_score(y, probs, multi_class="ovr"):.4f}\n')
print(report)
print(cm)
with open(f'{OUT}/released_checkpoint_test.txt', 'w') as f:
    f.write(f'accuracy {acc:.4f}\n\n{report}\n{cm}\n')
print('\nRESULT: PASS (close to reported number)' if abs(acc - 0.781) < 0.02
      else '\nRESULT: CHECK (accuracy is far from 78.1%, send this to Claude)')

# ---------- PART 3: timing ----------
print('\n========== PART 3: TRAINING SPEED ==========')
BS, EPOCHS, N_STEPS, WARMUP = 24, 100, 100, 10
model = MedViT_tiny(num_classes=len(CLASSES)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, betas=(0.9, 0.999), weight_decay=0.05)
loss_fn = nn.CrossEntropyLoss()
loader = DataLoader(ds['train'], batch_size=BS, shuffle=True, num_workers=2, drop_last=True)
model.train()
for i, (x, yb) in enumerate(loader):
    if i == WARMUP:
        torch.cuda.synchronize(); t0 = time.time()
    x, yb = x.to(device), yb.view(-1).long().to(device)
    opt.zero_grad(); loss = loss_fn(model(x), yb); loss.backward(); opt.step()
    if i == WARMUP + N_STEPS - 1:
        break
torch.cuda.synchronize()
sec_per_step = (time.time() - t0) / N_STEPS
t1 = time.time(); predict(model, ds['val']); val_sec = time.time() - t1
epoch_min = (sec_per_step * len(loader) + val_sec) / 60
run_hr = epoch_min * EPOCHS / 60
summary = f'''GPU: {torch.cuda.get_device_name(0)}
Seconds per training step (batch {BS}): {sec_per_step:.3f}
Steps per epoch: {len(loader)}
Validation pass: {val_sec:.0f} s
Estimated minutes per epoch: {epoch_min:.1f}
Estimated hours per {EPOCHS}-epoch run: {run_hr:.1f}
Estimated hours for ~14 runs: {14 * run_hr:.0f}
Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB'''
print(summary)
with open(f'{OUT}/timing.txt', 'w') as f:
    f.write(summary)
print('\n>>> ALL DONE. Copy everything above from "PART 1" down and send it to Claude.')

## Cell 5: Run the smoke test (15–25 minutes)
Downloads the data and the model, checks accuracy, and measures speed.

In [ ]:
# ===== CELL 5 =====
!/content/py311/bin/python /content/smoke_test.py

## Finished
Copy all the output of **Cell 5** (from PART 1 to ALL DONE) and send it to Claude.
If any cell shows a red error and no DONE line, send a screenshot of the end of the error.